# Dự báo Doanh thu Sản phẩm
## 5. Trích xuất & Tiền xử lý Đặc trưng (Feature Engineering)

Các bước thực hiện:
1. Phân tách tập dữ liệu thành Train và Test (tỷ lệ 80/20, `random_state=42`) để tránh rò rỉ dữ liệu.
2. Trích xuất đặc trưng thời gian: `Month` (Tháng dưới dạng chuỗi), `DayOfWeek` (Ngày trong tuần), và `Is_Weekend` (Biến nhị phân cuối tuần).
3. Xóa cột ngày gốc `invoice_date`.
4. Mã hóa các biến danh mục phân loại dạng chữ bằng `OneHotEncoder`.
5. Chuẩn hóa thang đo cột tuổi bằng `StandardScaler`.
6. Lưu trữ các ma trận dữ liệu huấn luyện và các bộ chuyển đổi tiền xử lý.

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

CLEANED_DATA_PATH = r'../data/preprocessed-data/preprocessed_customer_shopping_data.csv'
data = pd.read_csv(CLEANED_DATA_PATH, parse_dates=['invoice_date'])
print('Đã nạp dữ liệu sạch. Kích thước:', data.shape)

### 5.1 Phân tách Train-Test Split

In [ ]:
X = data.drop(columns=['Sales_Revenue'])
y = data['Sales_Revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Kích thước ma trận Train:', X_train.shape)
print('Kích thước ma trận Test :', X_test.shape)

### 5.2 Trích xuất đặc trưng thời gian

In [ ]:
def extract_time_features(df):
    df_copy = df.copy()
    df_copy['Month'] = df_copy['invoice_date'].dt.month.astype(str)
    df_copy['DayOfWeek'] = df_copy['invoice_date'].dt.day_name()
    df_copy['Is_Weekend'] = df_copy['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)
    df_copy = df_copy.drop(columns=['invoice_date'])
    return df_copy

X_train = extract_time_features(X_train)
X_test = extract_time_features(X_test)
print('Các cột dữ liệu hiện có:', list(X_train.columns))

### 5.3 Mã hóa One-Hot (One-Hot Encoding)
Các cột phân loại sẽ được mã hóa One-Hot. Encoder chỉ được fit trên tập Train để tránh rò rỉ dữ liệu.

In [ ]:
categorical_cols = ['gender', 'category', 'payment_method', 'shopping_mall', 'Month', 'DayOfWeek']
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoder.fit(X_train[categorical_cols])

X_train_encoded = encoder.transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

feature_names = encoder.get_feature_names_out(categorical_cols)

X_train_enc_df = pd.DataFrame(X_train_encoded, columns=feature_names, index=X_train.index)
X_test_enc_df = pd.DataFrame(X_test_encoded, columns=feature_names, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=categorical_cols), X_train_enc_df], axis=1)
X_test = pd.concat([X_test.drop(columns=categorical_cols), X_test_enc_df], axis=1)

print('Kích thước dữ liệu sau khi mã hóa One-Hot:', X_train.shape)

### 5.4 Chuẩn hóa thang đo đặc trưng tuổi (Standard Scaling)
Chúng ta chuẩn hóa thang đo của cột tuổi về phân phối chuẩn để hỗ trợ mô hình tuyến tính nếu cần.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[['age']])

X_train['age'] = scaler.transform(X_train[['age']])
X_test['age'] = scaler.transform(X_test[['age']])

X_train['age'].head()

### 5.5 Lưu trữ dữ liệu huấn luyện và test sạch

In [ ]:
out_dir = r'../data/ready_for_train'
os.makedirs(out_dir, exist_ok=True)

joblib.dump(X_train, os.path.join(out_dir, 'X_train.pkl'))
joblib.dump(X_test, os.path.join(out_dir, 'X_test.pkl'))
joblib.dump(y_train, os.path.join(out_dir, 'y_train.pkl'))
joblib.dump(y_test, os.path.join(out_dir, 'y_test.pkl'))

joblib.dump(encoder, os.path.join(out_dir, 'onehot_encoder.pkl'))
joblib.dump(scaler, os.path.join(out_dir, 'standard_scaler.pkl'))

print('Đã lưu thành công toàn bộ ma trận dữ liệu và mô hình tiền xử lý.')